# Demo 3: LLM API & Prompt Engineering

Calling an LLM via API: send a prompt, get structured results back. The techniques here apply directly to the assignment's `extractor.py`.

## Setup

In [1]:
%pip install -q openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# OpenRouter is an aggregator — same openai SDK, different base_url
if os.environ.get("OPENROUTER_API_KEY"):
    client = OpenAI(
        api_key=os.environ["OPENROUTER_API_KEY"],
        base_url="https://openrouter.ai/api/v1",
    )
    MODEL = "openai/gpt-4o-mini"
    print("Using OpenRouter")
elif os.environ.get("OPENAI_API_KEY"):
    client = OpenAI()
    MODEL = "gpt-4o-mini"
    print("Using OpenAI directly")
else:
    raise ValueError("Set OPENROUTER_API_KEY or OPENAI_API_KEY in a .env file")

Using OpenRouter


We'll use the same clinical note throughout so we can compare techniques directly. This is Note 1 from the assignment's `clinical_notes.txt`.

In [3]:
note = """Patient is a 58-year-old male presenting with chest pain radiating to the left arm.
Blood pressure 145/92 mmHg, heart rate 88 bpm. Troponin elevated at 0.8 ng/mL.
ECG shows ST elevation in leads V1-V4. Patient started on aspirin 325mg and
heparin drip. Diagnosis: Acute ST-elevation myocardial infarction (STEMI)."""

print(note)

Patient is a 58-year-old male presenting with chest pain radiating to the left arm.
Blood pressure 145/92 mmHg, heart rate 88 bpm. Troponin elevated at 0.8 ng/mL.
ECG shows ST elevation in leads V1-V4. Patient started on aspirin 325mg and
heparin drip. Diagnosis: Acute ST-elevation myocardial infarction (STEMI).


A reusable helper for calling the LLM. This matches the `call_llm` function already provided in the assignment's `extractor.py`.

In [4]:
def call_llm(prompt, system="You are a medical information extraction assistant.", temperature=0):
    """Send a prompt to the LLM and return the response text."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        temperature=temperature,
        max_tokens=500,
    )
    return response.choices[0].message.content

## Zero-Shot Prompting

Zero-shot: describe the task with no examples. Start here; add complexity only if needed.

In [5]:
zero_shot_prompt = f"""Extract the primary diagnosis from this clinical note.

Clinical Note:
{note}

Diagnosis:"""

response = call_llm(zero_shot_prompt)
print(response)

Acute ST-elevation myocardial infarction (STEMI)


The model returns free text. That's fine for reading but hard to process programmatically — there's no guaranteed format, no machine-readable structure.

## Few-Shot Prompting

Few-shot provides examples before the actual input. The model infers the pattern from examples rather than instructions alone.

In [6]:
one_shot_prompt = f"""Extract the primary diagnosis from clinical notes.

Example:
Note: "65-year-old female with polyuria, polydipsia, fasting glucose 285 mg/dL, HbA1c 9.2%."
Diagnosis: Type 2 Diabetes Mellitus (poorly controlled)

Now extract the diagnosis:
Note: "{note}"
Diagnosis:"""

response = call_llm(one_shot_prompt)
print(response)

Acute ST-elevation myocardial infarction (STEMI)


In [7]:
two_shot_prompt = f"""Extract the primary diagnosis from clinical notes.

Example 1:
Note: "65-year-old female with polyuria, polydipsia, fasting glucose 285 mg/dL, HbA1c 9.2%."
Diagnosis: Type 2 Diabetes Mellitus (poorly controlled)

Example 2:
Note: "42-year-old male with productive cough, fever to 101.5F, right lower lobe infiltrate on X-ray."
Diagnosis: Community-acquired pneumonia

Now extract the diagnosis:
Note: "{note}"
Diagnosis:"""

response = call_llm(two_shot_prompt)
print(response)

Acute ST-elevation myocardial infarction (STEMI)


Few-shot tends to produce more consistent formatting because the model mirrors the style of the examples.

## Structured Outputs (JSON Extraction)

Free-text responses are hard to use downstream. If you want to write results to a database, pass them to another function, or validate them, you need a schema. The standard approach: put the schema directly in the prompt.

Note the double braces `{{` / `}}` — Python f-strings use single braces for variable interpolation, so literal braces in JSON must be doubled.

In [8]:
schema_prompt = f"""Extract structured information from this clinical note.
Return ONLY a JSON object with exactly these fields:

{{
  "diagnosis": "<primary diagnosis as a string>",
  "medications": ["<list of medications mentioned>"],
  "lab_values": {{"<test name>": "<value with units>"}},
  "confidence": <float 0.0 to 1.0>
}}

Clinical Note:
{note}"""

response = call_llm(schema_prompt)
print(response)

{
  "diagnosis": "Acute ST-elevation myocardial infarction (STEMI)",
  "medications": ["aspirin 325mg", "heparin drip"],
  "lab_values": {"Troponin": "0.8 ng/mL"},
  "confidence": 0.95
}


The response is usually valid JSON, but LLMs sometimes wrap it in markdown code fences or add commentary. We need a parser that handles all three cases.

In [9]:
def parse_json_response(text):
    """Extract JSON from an LLM response, handling markdown code fences."""
    # Strategy 1: direct parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Strategy 2: strip markdown code fences
    if "```" in text:
        start = text.find("```")
        end = text.rfind("```")
        if start != end:
            block = text[start:end]
            lines = block.split("\n")
            # Drop the opening ``` line (may include language label like ```json)
            block = "\n".join(lines[1:])
            try:
                return json.loads(block)
            except json.JSONDecodeError:
                pass

    # Strategy 3: find outermost { ... }
    start = text.find("{")
    end = text.rfind("}") + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass

    return None


parsed = parse_json_response(response)
if parsed:
    print(json.dumps(parsed, indent=2))
else:
    print("Could not parse JSON. Raw response:")
    print(response)

{
  "diagnosis": "Acute ST-elevation myocardial infarction (STEMI)",
  "medications": [
    "aspirin 325mg",
    "heparin drip"
  ],
  "lab_values": {
    "Troponin": "0.8 ng/mL"
  },
  "confidence": 0.95
}


Now validate that the parsed result has the fields we asked for.

In [10]:
def validate_response(response):
    """Check that the response has all required fields."""
    if not isinstance(response, dict):
        return False
    required = {"diagnosis", "medications", "lab_values", "confidence"}
    return required.issubset(response.keys())


if parsed:
    print(f"Valid: {validate_response(parsed)}")
    print(f"Diagnosis:   {parsed.get('diagnosis')}")
    print(f"Medications: {parsed.get('medications')}")
    print(f"Lab values:  {parsed.get('lab_values')}")
    print(f"Confidence:  {parsed.get('confidence')}")

Valid: True
Diagnosis:   Acute ST-elevation myocardial infarction (STEMI)
Medications: ['aspirin 325mg', 'heparin drip']
Lab values:  {'Troponin': '0.8 ng/mL'}
Confidence:  0.95


## Few-Shot + JSON

When zero-shot JSON extraction produces inconsistent schemas (wrong field names, missing fields), adding a complete JSON example anchors the format.

In [11]:
few_shot_json_prompt = f"""Extract structured information from clinical notes. Return JSON only.

Example:
Note: "65-year-old female with polyuria, polydipsia. Fasting glucose 285 mg/dL, HbA1c 9.2%.
Taking metformin 1000mg BID and lisinopril 10mg daily."

Output:
{{
  "diagnosis": "Type 2 Diabetes Mellitus (poorly controlled)",
  "medications": ["metformin 1000mg BID", "lisinopril 10mg daily"],
  "lab_values": {{"fasting_glucose": "285 mg/dL", "HbA1c": "9.2%"}},
  "confidence": 0.95
}}

Now extract from this note:
Note: "{note}"

Output:"""

response = call_llm(few_shot_json_prompt)
parsed_few = parse_json_response(response)
print(json.dumps(parsed_few, indent=2) if parsed_few else response)

{
  "diagnosis": "Acute ST-elevation myocardial infarction (STEMI)",
  "medications": [
    "aspirin 325mg",
    "heparin drip"
  ],
  "lab_values": {
    "blood_pressure": "145/92 mmHg",
    "heart_rate": "88 bpm",
    "troponin": "0.8 ng/mL"
  },
  "ecg_findings": "ST elevation in leads V1-V4",
  "confidence": 0.95
}


The example anchors field names, value formats (units in lab values, full dosing in medications). This is the pattern the assignment's `build_prompt(few_shot=True)` should implement.

## Chain-of-Thought Prompting

Chain-of-thought asks the model to reason step by step before producing the answer. More tokens, slower, but often more accurate for complex or ambiguous cases.

In [12]:
cot_prompt = f"""Extract structured data from the clinical note below.

First, reason through the key elements step by step:
1. What is the primary diagnosis?
2. What medications are mentioned (include dose and frequency if given)?
3. What lab values are reported (include units)?
4. How confident are you in this extraction (0.0-1.0)?

Then produce the final JSON:
{{
  "diagnosis": "<primary diagnosis>",
  "medications": ["<list>"],
  "lab_values": {{"<name>": "<value>"}},
  "confidence": <float>
}}

Clinical Note:
{note}"""

response = call_llm(cot_prompt)
print(response)

Let's analyze the clinical note step by step:

1. **Primary Diagnosis**: The note clearly states the diagnosis as "Acute ST-elevation myocardial infarction (STEMI)".

2. **Medications**: The medications mentioned are:
   - Aspirin: 325 mg (dose is specified, but frequency is not mentioned)
   - Heparin drip (no specific dose or frequency provided)

3. **Lab Values**: The lab value reported is:
   - Troponin: 0.8 ng/mL

4. **Confidence in Extraction**: I am confident in this extraction as the information is clearly stated in the clinical note. Therefore, I would rate my confidence at 1.0.

Now, I will compile the extracted information into the requested JSON format:

```json
{
  "diagnosis": "Acute ST-elevation myocardial infarction (STEMI)",
  "medications": ["Aspirin 325mg", "Heparin drip"],
  "lab_values": {"Troponin": "0.8 ng/mL"},
  "confidence": 1.0
}
```


In [13]:
parsed_cot = parse_json_response(response)
if parsed_cot:
    print("\nExtracted JSON:")
    print(json.dumps(parsed_cot, indent=2))


Extracted JSON:
{
  "diagnosis": "Acute ST-elevation myocardial infarction (STEMI)",
  "medications": [
    "Aspirin 325mg",
    "Heparin drip"
  ],
  "lab_values": {
    "Troponin": "0.8 ng/mL"
  },
  "confidence": 1.0
}


The reasoning is visible (useful for debugging) and the same `parse_json_response` function works — it finds the JSON block at the end regardless of the preceding text.

## Batch Extraction (Assignment Preview)

The assignment asks you to run extraction on all four notes in `clinical_notes.txt`. Here's the end-to-end workflow.

In [14]:
def load_notes(filepath):
    """Load notes from the assignment's clinical_notes.txt format."""
    with open(filepath) as f:
        content = f.read()
    sections = content.split("## Note")
    return [s.split("\n", 1)[1].strip() for s in sections[1:] if s.strip()]


# Find the notes file (works from demo/ or repo root)
candidates = [
    "../assignment/clinical_notes.txt",
    "lectures/07/assignment/clinical_notes.txt",
]
notes_path = next((p for p in candidates if os.path.exists(p)), None)

if notes_path:
    notes = load_notes(notes_path)
    print(f"Loaded {len(notes)} notes")
    for i, n in enumerate(notes, 1):
        print(f"\n--- Note {i} ---\n{n[:80]}...")
else:
    print("clinical_notes.txt not found. Run from lectures/07/demo/ or repo root.")

Loaded 4 notes

--- Note 1 ---
Patient is a 58-year-old male presenting with chest pain radiating to the left a...

--- Note 2 ---
65-year-old female with history of type 2 diabetes presents with polyuria and 
p...

--- Note 3 ---
42-year-old male with productive cough for 5 days, fever to 101.5F. Chest X-ray ...

--- Note 4 ---
28-year-old female with severe headache, photophobia, and neck stiffness. 
Tempe...


In [15]:
if notes_path:
    results = []
    for i, note_text in enumerate(notes, 1):
        print(f"\nProcessing Note {i}...")
        prompt = f"""Extract structured information from clinical notes. Return JSON only.

Example:
Note: "65-year-old female with polyuria, polydipsia. Fasting glucose 285 mg/dL, HbA1c 9.2%.
Taking metformin 1000mg BID and lisinopril 10mg daily."

Output:
{{
  "diagnosis": "Type 2 Diabetes Mellitus (poorly controlled)",
  "medications": ["metformin 1000mg BID", "lisinopril 10mg daily"],
  "lab_values": {{"fasting_glucose": "285 mg/dL", "HbA1c": "9.2%"}},
  "confidence": 0.95
}}

Now extract from this note:
Note: "{note_text}"

Output:"""
        response = call_llm(prompt)
        parsed = parse_json_response(response)
        if parsed and validate_response(parsed):
            results.append({"note_id": i, **parsed})
            print(f"  Diagnosis: {parsed['diagnosis']}")
            print(f"  Confidence: {parsed['confidence']}")
        else:
            print(f"  Extraction failed. Raw: {response[:100]}")

    print(f"\n{len(results)}/{len(notes)} notes extracted successfully")


Processing Note 1...


  Diagnosis: Acute ST-elevation myocardial infarction (STEMI)
  Confidence: 0.95

Processing Note 2...


  Diagnosis: Type 2 Diabetes Mellitus (poorly controlled)
  Confidence: 0.95

Processing Note 3...


  Diagnosis: Community-acquired pneumonia
  Confidence: 0.95

Processing Note 4...


  Diagnosis: Bacterial meningitis
  Confidence: 0.95

4/4 notes extracted successfully
